# Studio setup + Camber pull cho V6.5-12
Chạy từ trên xuống trong kernel mặc định của Studio, không tạo Conda environment.
Notebook này chỉ tải/kiểm tra, không train hoặc ghi lên Stash.
Cần **dataset v6p5_roboflow_ablation đã build** và **best.pt của run 20261009T024019Z_roboflow**.
Dataset compose đơn thuần không thay thế được dataset Roboflow. Notebook v6p5-11 không upload các artifact này; nếu Stash chưa có, cần upload từ máy cũ.


In [1]:
# 0. Tìm project và cấu hình
from pathlib import Path
import os, sys, subprocess, shutil, json, hashlib

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Uav-Powerline-Defect-Detection"]
PROJECT = next((p for p in candidates if (p / "pyproject.toml").is_file()
                and (p / "legacy/notebooks/v6p5").is_dir()), None)
if PROJECT is None:
    raise RuntimeError("Đặt PROJECT thành đường dẫn repository trên Studio.")
DATASET = Path.home() / "datasets/v6p5_roboflow_ablation"
CACHE = PROJECT / "data/cache/studio_v6p5_pull"
MODEL = PROJECT / "legacy/notebooks/v6p5/yolo11n.pt"
PREV_BEST = PROJECT / "artifacts/v6p5_train_local/20261009T024019Z_roboflow/runs/yolo11n_v6p5_roboflow_ablation_640/weights/best.pt"
STASH_ROOT = "stash://bunpmc/projects/yolov11sdi"
CACHE.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT)
print("Dataset:", DATASET)
print("Disk free GB:", round(shutil.disk_usage(DATASET.parent if DATASET.parent.exists() else Path.home()).free / 2**30, 2))


Project: /teamspace/studios/this_studio/Uav-Powerline-Defect-Detection
Dataset: /teamspace/studios/this_studio/datasets/v6p5_roboflow_ablation
Disk free GB: 313.15


In [2]:
# 1. Dependencies trong environment hiện tại (giữ PyTorch của Studio)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{PROJECT}[train,notebook]"])
# Nếu torch/ultralytics đã được import trước khi cài, restart kernel rồi chạy lại.


Obtaining file:///teamspace/studios/this_studio/Uav-Powerline-Defect-Detection
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for yolov11sdi (pyproject.toml): started
  Building editable for yolov11sdi (pyproject.toml): finished with status 'done'
  Created wheel for yolov11sdi: filename=yolov11sdi-0.2.0-0.editable-py3-none-any.whl size=1550 sha256=2940183f5e173ed27d3d1f132dd6b3b581a5ad9e002645c4fbfd6543420a81fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-x29d7ntf/wheels/9a/0b/00/aa535487e758262def77b075e1291c382c0acd04

0

In [3]:
# 2. Camber CLI + auth (không in/lưu API key)
from getpass import getpass
from urllib.request import urlopen

# Load .env without printing secrets or overriding Studio environment.
subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
from dotenv import load_dotenv
load_dotenv(PROJECT / ".env", override=False)
sys.path.insert(0, str(PROJECT / "src"))
from yolov11sdi.camber import resolve_camber_bin
CAMBER = resolve_camber_bin()
if not CAMBER:
    installer = CACHE / "camber-install-v2.sh"
    print("Installing Camber CLI...")
    with urlopen("https://cli.cambercloud.com/install-v2.sh", timeout=60) as response:
        installer.write_bytes(response.read())
    subprocess.check_call(["bash", str(installer)])
    CAMBER = resolve_camber_bin()
    if not CAMBER:
        raise RuntimeError("Installer hoàn tất nhưng chưa tìm thấy CLI. Đặt CAMBER_BIN trong .env thành đường dẫn binary camber.")
if not os.environ.get("CAMBER_API_KEY"):
    probe = subprocess.run([CAMBER, "me"], capture_output=True, text=True)
    if probe.returncode != 0 or "authentication error" in (probe.stdout + probe.stderr).lower():
        key = getpass("CAMBER_API_KEY (ẩn, chỉ giữ trong kernel): ").strip()
        if not key:
            raise RuntimeError("Cần CAMBER_API_KEY hoặc camber login trong terminal.")
        os.environ["CAMBER_API_KEY"] = key
        del key

def run_camber(*args):
    r = subprocess.run([CAMBER, *args], capture_output=True, text=True, env=os.environ.copy())
    if r.returncode:
        raise RuntimeError((r.stderr or r.stdout)[:4000])
    return r.stdout
print("Camber CLI:", CAMBER)


Camber CLI: /teamspace/studios/this_studio/.camber/bin/camber


In [4]:
# 3. Liệt kê đúng thư mục transfer từ máy cũ
TRANSFER_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_12_transfer/20261010"
listing = run_camber("stash", "ls", "-r", TRANSFER_ROOT)
(CACHE / "transfer_listing.txt").write_text(listing)
print(listing)
print("Điền URI file đầy đủ ở cell 4, dùng TRANSFER_ROOT + tên file tương đối trong listing.")



Name                         Type                Size
----                         ----                ----
TRANSFER.md                  file                2.2KB
v6p5_12_resume_pack.tar.gz   file                55.3MB
v6p5_roboflow_ablation.tar   file                5.4GB


Điền URI file đầy đủ ở cell 4, dùng TRANSFER_ROOT + tên file tương đối trong listing.


In [5]:
print((CACHE / "datasets_listing.txt").read_text())

remote = (
    f"{STASH_ROOT}/checkpoints/v6p5_data_ablation/"
    "v6p5_y11n_ablation_761a2f41098d"
)
print(run_camber("stash", "ls", "-r", remote))


Name                                                                                                           Type                Size
----                                                                                                           ----                ----
curated_v4_7class.meta.json                                                                                    file                456B
curated_v4_7class.tar.gz                                                                                       file                793.3MB
curated_v5_clean.meta.json                                                                                     file                582B
curated_v5_clean.tar.gz                                                                                        file                370.8MB
curated_v6.meta.json                                                                                           file                1.2KB
curated_v6.tar.gz                       

In [6]:
TRANSFER_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_12_transfer/20261010"
print(run_camber("stash", "ls", "-r", TRANSFER_ROOT))


Name                         Type                Size
----                         ----                ----
TRANSFER.md                  file                2.2KB
v6p5_12_resume_pack.tar.gz   file                55.3MB
v6p5_roboflow_ablation.tar   file                5.4GB




In [7]:
# 4. Transfer artifacts đã xác nhận từ listing
TRANSFER_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_12_transfer/20261010"
DATASET_REMOTE = f"{TRANSFER_ROOT}/v6p5_roboflow_ablation.tar"
PACK_REMOTE = f"{TRANSFER_ROOT}/v6p5_12_resume_pack.tar.gz"
DATASET_SHA256 = "abfe19fba5cf70d4ca6f822417c2dc9eecb9da053688fa66735dc3040236743f"
PACK_SHA256 = "9e1cbc24b91b0866979e3f67c774973cf7c6a6399f7920f7fa6399fb8a195aea"


In [8]:
print((CACHE / "TRANSFER.md").read_text())

print("\nCác file .pt trong pack:")
for p in sorted(PACK_DIR.rglob("*.pt")):
    print(p.relative_to(PACK_DIR))

# V6.5-12 transfer bundle

Remote root:

`stash://bunpmc/projects/yolov11sdi/datasets/v6p5_12_transfer/20261010`

## Files

- `v6p5_roboflow_ablation.tar`
  - 5.4 GiB, 41,222 TAR entries.
  - SHA256: `abfe19fba5cf70d4ca6f822417c2dc9eecb9da053688fa66735dc3040236743f`
- `v6p5_12_resume_pack.tar.gz`
  - SHA256: `9e1cbc24b91b0866979e3f67c774973cf7c6a6399f7920f7fa6399fb8a195aea`
  - Contains `yolo11n.pt`, the MuSGD comparison `best.pt`, AdamW
    `last.pt`/`best.pt` through epoch 29, training metadata, and the V6.5-12
    notebook.

## Restore on a new WSL machine

Download both archives from the remote root. From the repository root:

```bash
mkdir -p /home/intern/datasets
tar -xf v6p5_roboflow_ablation.tar -C /home/intern/datasets

mkdir -p /tmp/v6p5_12_restore
tar -xzf v6p5_12_resume_pack.tar.gz -C /tmp/v6p5_12_restore

cp /tmp/v6p5_12_restore/checkpoints/yolo11n.pt \
  legacy/notebooks/v6p5/yolo11n.pt

mkdir -p artifacts/v6p5_train_local/20261009T024019Z_roboflow/runs/yolo11n_v6p5_robof

NameError: name 'PACK_DIR' is not defined

In [ ]:
# 5. Pull hướng dẫn + resume pack + dataset (~5.4 GB)
import tarfile
from yolov11sdi.camber import CamberClient
from yolov11sdi.paths import ProjectPaths
cam = CamberClient(ProjectPaths.detect(project_root=PROJECT), STASH_ROOT, binary=CAMBER)
transfer_md = cam.fetch(f"{TRANSFER_ROOT}/TRANSFER.md", CACHE / "TRANSFER.md", reuse=False)
print(transfer_md.read_text())
pack = cam.fetch(PACK_REMOTE, CACHE / "v6p5_12_resume_pack.tar.gz",
                 expected_sha256=PACK_SHA256)
PACK_DIR = CACHE / "resume_pack"
PACK_DIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(pack, "r:gz") as t:
    t.extractall(PACK_DIR, filter="data")
weights = sorted(PACK_DIR.rglob("*.pt"))
print("Checkpoint/model trong resume pack:")
for weight in weights:
    print(weight.relative_to(PACK_DIR))

# Chỉ tự chọn khi có đúng một ứng viên, tránh lấy nhầm checkpoint AdamW.
model_candidates = [p for p in weights if p.name == "yolo11n.pt"]
best_candidates = [p for p in weights if p.name == "roboflow_musgd_best.pt"]
if not best_candidates:
    best_candidates = [p for p in weights if p.name == "best.pt"
                   and "20261009T024019Z_roboflow" in p.as_posix()]
if not best_candidates:
    best_candidates = [p for p in weights if p.name == "best.pt"
                       and "yolo11n_v6p5_roboflow_ablation_640" in p.as_posix()]
# Nếu pack đổi tên/flatten, xem TRANSFER.md rồi đặt đường dẫn rõ ràng tại đây.
MODEL_IN_PACK = PACK_DIR / "checkpoints/yolo11n.pt"
MUSGD_BEST_IN_PACK = PACK_DIR / "checkpoints/roboflow_musgd_best.pt"
model_src = Path(MODEL_IN_PACK) if MODEL_IN_PACK else (model_candidates[0] if len(model_candidates) == 1 else None)
best_src = Path(MUSGD_BEST_IN_PACK) if MUSGD_BEST_IN_PACK else (best_candidates[0] if len(best_candidates) == 1 else None)
if model_src is None or best_src is None:
    raise RuntimeError("Chưa xác định duy nhất model gốc/best MuSGD. Xem TRANSFER.md và listing ở trên, điền MODEL_IN_PACK và MUSGD_BEST_IN_PACK trong cell này.")
for src, dst in [(model_src, MODEL), (best_src, PREV_BEST)]:
    if not src.is_file():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and hashlib.sha256(dst.read_bytes()).digest() != hashlib.sha256(src.read_bytes()).digest():
        raise RuntimeError(f"File đích khác checkpoint transfer: {dst}; kiểm tra trước khi thay thế.")
    if not dst.exists():
        shutil.copy2(src, dst)
    print("Ready:", dst)
archive = cam.fetch(DATASET_REMOTE, CACHE / "v6p5_roboflow_ablation.tar",
                    expected_sha256=DATASET_SHA256 or None)
print("Downloaded:", archive)


# V6.5-12 transfer bundle

Remote root:

`stash://bunpmc/projects/yolov11sdi/datasets/v6p5_12_transfer/20261010`

## Files

- `v6p5_roboflow_ablation.tar`
  - 5.4 GiB, 41,222 TAR entries.
  - SHA256: `abfe19fba5cf70d4ca6f822417c2dc9eecb9da053688fa66735dc3040236743f`
- `v6p5_12_resume_pack.tar.gz`
  - SHA256: `9e1cbc24b91b0866979e3f67c774973cf7c6a6399f7920f7fa6399fb8a195aea`
  - Contains `yolo11n.pt`, the MuSGD comparison `best.pt`, AdamW
    `last.pt`/`best.pt` through epoch 29, training metadata, and the V6.5-12
    notebook.

## Restore on a new WSL machine

Download both archives from the remote root. From the repository root:

```bash
mkdir -p /home/intern/datasets
tar -xf v6p5_roboflow_ablation.tar -C /home/intern/datasets

mkdir -p /tmp/v6p5_12_restore
tar -xzf v6p5_12_resume_pack.tar.gz -C /tmp/v6p5_12_restore

cp /tmp/v6p5_12_restore/checkpoints/yolo11n.pt \
  legacy/notebooks/v6p5/yolo11n.pt

mkdir -p artifacts/v6p5_train_local/20261009T024019Z_roboflow/runs/yolo11n_v6p5_robof

In [ ]:
# 6. Extract an toàn vào staging, không trộn dataset cũ
import tarfile, zipfile, tempfile
import yaml
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
marker = DATASET / ".camber_source.json"
source = {"uri": DATASET_REMOTE, "archive_sha256": hashlib.file_digest(archive.open("rb"), "sha256").hexdigest()}
if DATASET.exists():
    if not marker.is_file() or json.loads(marker.read_text()) != source:
        raise RuntimeError(f"Dataset đã tồn tại với provenance khác/không rõ: {DATASET}. Chọn DATASET mới hoặc kiểm tra thủ công.")
    print("Reuse dataset:", DATASET)
else:
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="v6p5_extract_", dir=DATASET.parent) as tmp:
        stage = Path(tmp)
        if tarfile.is_tarfile(archive):
            with tarfile.open(archive) as t:
                t.extractall(stage, filter="data")
        elif zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as z:
                for item in z.infolist():
                    target = (stage / item.filename).resolve()
                    if not target.is_relative_to(stage.resolve()):
                        raise ValueError("Unsafe ZIP path")
                    if (item.external_attr >> 16) & 0o170000 == 0o120000:
                        raise ValueError("ZIP symlink unsupported")
                z.extractall(stage)
        else:
            raise ValueError("Dataset phải là tar/tar.gz/zip")
        roots = []
        for yp in stage.rglob("data.yaml"):
            cfg = yaml.safe_load(yp.read_text())
            names = cfg.get("names", [])
            names = [names[k] for k in sorted(names, key=int)] if isinstance(names, dict) else names
            if names == NAMES:
                roots.append(yp.parent)
        if len(roots) != 1:
            raise RuntimeError(f"Cần đúng một dataset 6 lớp; tìm thấy {len(roots)}")
        root = roots[0]
        # Dataset v6p5-11 dùng images/{train,val}, labels/{train,val}.
        for split in ["train", "val"]:
            for kind in ["images", "labels"]:
                if not (root / kind / split).is_dir():
                    raise FileNotFoundError(root / kind / split)
        shutil.move(str(root), str(DATASET))
    marker.write_text(json.dumps(source, indent=2))
# Chỉ đổi đường dẫn runtime, giữ taxonomy/split.
yp = DATASET / "data.yaml"
cfg = yaml.safe_load(yp.read_text())
cfg.update(path=str(DATASET.resolve()), train="images/train", val="images/val")
yp.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Ready:", yp)


Ready: /teamspace/studios/this_studio/datasets/v6p5_roboflow_ablation/data.yaml


In [9]:
# 7. Dataset + GPU gate trước khi train
from pathlib import Path
import sys
if "DATASET" not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Uav-Powerline-Defect-Detection"]
    PROJECT = next((p for p in candidates if (p / "pyproject.toml").is_file()
                    and (p / "legacy/notebooks/v6p5").is_dir()), None)
    if PROJECT is None:
        raise RuntimeError("Chạy cell 0 để cấu hình PROJECT trước.")
    DATASET = Path.home() / "datasets/v6p5_roboflow_ablation"
    MODEL = PROJECT / "legacy/notebooks/v6p5/yolo11n.pt"
    PREV_BEST = PROJECT / "artifacts/v6p5_train_local/20261009T024019Z_roboflow/runs/yolo11n_v6p5_roboflow_ablation_640/weights/best.pt"
if not (DATASET / "data.yaml").is_file():
    raise RuntimeError(f"Chưa thấy dataset tại {DATASET}. Chạy setup/pull cell 0–6 trước; nếu đã đổi máy, kiểm tra ổ lưu dataset.")

import torch, ultralytics
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
for split in ["train", "val"]:
    images = [p for p in (DATASET / "images" / split).iterdir() if p.suffix.lower() in EXTS]
    missing = [p.name for p in images if not (DATASET / "labels" / split / (p.stem + ".txt")).is_file()]
    if not images or missing:
        raise RuntimeError(f"{split}: images={len(images)}, thiếu labels={len(missing)}")
    if split == "val" and len(images) != 2272:
        raise RuntimeError(f"Frozen VAL phải có 2272 ảnh, hiện có {len(images)}")
    print(split, len(images))
assert MODEL.is_file() and PREV_BEST.is_file()
assert ultralytics.__version__ == "8.4.172", ultralytics.__version__
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Studio chưa nhận CUDA; chọn GPU Studio/kernel phù hợp"
print("GPU:", torch.cuda.get_device_name(0))
print("Copy các dòng sau vào CONFIG của v6p5-12:")
for name in ["PROJECT", "DATASET", "MODEL", "PREV_BEST"]:
    print(f'{name} = Path({str(globals()[name])!r})')


train 18334
val 2272
Python: 3.14.7 PyTorch: 2.14.0+cu130
GPU: NVIDIA L4
Copy các dòng sau vào CONFIG của v6p5-12:
PROJECT = Path('/teamspace/studios/this_studio/Uav-Powerline-Defect-Detection')
DATASET = Path('/teamspace/studios/this_studio/datasets/v6p5_roboflow_ablation')
MODEL = Path('/teamspace/studios/this_studio/Uav-Powerline-Defect-Detection/legacy/notebooks/v6p5/yolo11n.pt')
PREV_BEST = Path('/teamspace/studios/this_studio/Uav-Powerline-Defect-Detection/artifacts/v6p5_train_local/20261009T024019Z_roboflow/runs/yolo11n_v6p5_roboflow_ablation_640/weights/best.pt')


## Nếu Stash chưa có dataset Roboflow
Trên **máy cũ** nơi còn dataset đã build, có thể đóng gói và upload bằng Camber CLI:
```bash
tar -czf v6p5_roboflow_ablation.tar.gz -C "$HOME/datasets" v6p5_roboflow_ablation
camber stash cp v6p5_roboflow_ablation.tar.gz stash://bunpmc/projects/yolov11sdi/datasets/v6p5_roboflow_ablation/20261009T024019Z_roboflow/v6p5_roboflow_ablation.tar.gz
```
Upload thêm `best.pt` của run `20261009T024019Z_roboflow` và file khởi tạo `yolo11n.pt` vào URI riêng rồi điền cell 4. Các lệnh trên chỉ là hướng dẫn, notebook không tự upload.
Sau khi cell 7 pass, mở v6p5-12, thay CONFIG bằng đường dẫn đã in rồi chạy từ trên xuống.
